# Modern Structural Pattern Matching (match-case)

## Core Mechanics & Theory

Introduced in Python 3.10 (PEP 634), match-case is not just a switch-case statement. It performs structural destructuring, type checking, and variable binding simultaneously in a single pass.

## Basic Syntax

```python
match subject:
    case <Pattern_1>:
        <action_1>
    case <Pattern_2> if <Guard_Condition>:
        <action_2>
    case _:
        <default_fallback_action>
```

## Switch vs. Match Conceptually

Yes — the easiest way to understand Python match-case is:

- **switch** usually asks: "Is this value equal to X?"
- **match** asks: "Does this value have this shape/pattern, and if so, what can I extract from it?"

## 1. Normal Switch Thinking

Imagine a typical switch:

```
switch day:
    case 1: "Monday"
    case 2: "Tuesday"
    case 3: "Wednesday"
```

It's basically:

```
value == 1?
value == 2?
value == 3?
```

Python doesn't traditionally have switch, but you can use match for this:

In [1]:
day = 2

match day:
    case 1:
        print("Monday")
    case 2:
        print("Tuesday")
    case 3:
        print("Wednesday")

Tuesday


## 2. Where Match Becomes Much More Powerful

Suppose you have:

```
point = (10, 20)
```

You can do:

```python
match point:
    case (0, 0):
        print("Origin")
    case (x, 0):
        print(f"On X axis: {x}")
    case (0, y):
        print(f"On Y axis: {y}")
    case (x, y):
        print(f"Point: {x}, {y}")
```

This is not just comparison.

When Python sees:

```python
case (x, y):
```

it means:

"Does point have the shape (something, something)? If yes, take those two values and bind them to x and y."

So:

```
point = (10, 20)
```

becomes conceptually:

```
x = 10
y = 20
```

That's structural destructuring + variable binding.

In [2]:
point = (10, 20)

match point:
    case (0, 0):
        print("Origin")
    case (x, 0):
        print(f"On X axis: {x}")
    case (0, y):
        print(f"On Y axis: {y}")
    case (x, y):
        print(f"Point: {x}, {y}")

Point: 10, 20


## 3. Type/Shape Checking

You can also match based on type:

```
value = 42
```

And write:

```python
match value:
    case int(x):
        print(f"Integer: {x}")
    case str(x):
        print(f"String: {x}")
```

If:

```
value = 42
```

Python recognizes it as an int and binds:

```
x = 42
```

So match can combine:

- **Pattern matching** → does it have this shape?
- **Type checking** → is it this type?
- **Variable binding** → extract values into variables

In [3]:
value = 42

match value:
    case int(x):
        print(f"Integer: {x}")
    case str(x):
        print(f"String: {x}")

Integer: 42


## 4. A Very Useful Real-World Example

Imagine an API response:

```
response = {
    "status": "error",
    "code": 404
}
```

You can write:

```python
match response:
    case {"status": "success", "data": data}:
        print("Got data:", data)
    case {"status": "error", "code": code}:
        print("Error code:", code)
    case _:
        print("Unknown response")
```

The important part is:

```python
case {"status": "error", "code": code}:
```

It means:

"Does this dictionary contain status="error" and a code? If yes, extract the code into code."

So you aren't merely saying:

```
response == something
```

You're describing the structure you expect.

In [4]:
response = {
    "status": "error",
    "code": 404
}

match response:
    case {"status": "success", "data": data}:
        print("Got data:", data)
    case {"status": "error", "code": code}:
        print("Error code:", code)
    case _:
        print("Unknown response")

Error code: 404


## The Mental Model to Remember

Think of switch as:

**"Which VALUE is this?"**

Think of Python match as:

**"What SHAPE/TYPE is this, and what values can I extract from it?"**

That's why structural pattern matching is a better name than "Python's switch."

## Comparison: if-elif vs. match-case
Handling a heterogeneous stream of instructions without match-case requires repetitive isinstance, len(), and dictionary key checks:

In [5]:
# Old Verbose Approach
if isinstance(cmd, dict) and cmd.get("type") == "SYNC":
    device_id = cmd.get("device")
    # ...
elif isinstance(cmd, list) and len(cmd) == 3 and cmd[0] == "KERNEL":
    name, threads = cmd[1], cmd[2]
    # ...

NameError: name 'cmd' is not defined

In [ ]:
# Modern Structural Pattern Matching
match cmd:
    case {"type": "SYNC", "device": int(dev_id)}:
        sync_device(dev_id)
    case ["KERNEL", str(name), int(threads)] if threads <= 1024:
        launch_kernel(name, threads)
    case _:
        raise ValueError(f"Malformed command: {cmd}")

Exercise 1: GPU Instruction Packet DecoderBuild a dispatch engine decode_instruction(packet) that matches incoming command structures:Input formats to support:("ALLOC", int_size) $\rightarrow$ Print "Allocating <size> bytes".("FREE", hex_address) $\rightarrow$ Print "Freeing pointer <hex_address>".("LAUNCH", kernel_name, (grid_x, grid_y), threads) $\rightarrow$ Print "Launching <kernel_name> on (<grid_x>, <grid_y>) with <threads> threads".("LAUNCH", kernel_name, threads) $\rightarrow$ Print "Launching <kernel_name> 1D grid with <threads> threads".Any sequence starting with "LAUNCH" where threads > 1024 $\rightarrow$ Print "Error: Thread block size exceeds maximum (1024)" using a guard clause (if).Catch-all _ $\rightarrow$ Raise ValueError(f"Unknown instruction: {packet}").

In [13]:
def decode_instructions(packet):
    match packet:
        case ("ALLOC",int(size)):
            print(f"Allocating the {size} bytes")
        
        case ("FREE",address):
            print(f"Freeing the pointer {address}")
            
        case ("LAUNCH",kernal_name,(grid_x,grid_y),threads):
            print(f"Launching the kernal {kernal_name} on ({grid_x,grid_y}) with {threads} threads")
            
        case ("LAUNCH",kernal_name,threads):
            print(f"Launching {kernal_name} with 1D threads {threads} threads")
            
        case ("LAUNCH",threads) if threads>1024:
            print("Error: thred blcok size exceeds the maximum 1024")

        case _:
            raise ValueError(f"Unkown instrcution: {packet}")
        
#--->
# hex(address) is invalid syntax in patterns: int(size) works because int is a type/class pattern. hex is a built-in function, not a type. For addresses, match the variable directly (address) or type-check with str(address).

# Order of cases (Guard placement): In match-case, Python evaluates top-to-bottom and stops at the first match.

# If you place case ("LAUNCH", kernel, threads) before the guard check if threads > 1024, the unguarded pattern matches first every time, and the thread limit check never runs.

# Rule: Always put narrower/guarded patterns above general ones.

# correct:
def decode_instructions(packet):
    match packet:
        case ("ALLOC", int(size)):
            print(f"Allocating {size} bytes")
        case ("FREE", str(address)):
            print(f"Freeing pointer {address}")
        # Guarded check placed BEFORE general LAUNCH matches
        case ("LAUNCH", kernel, _, int(threads)) if threads > 1024:
            print("Error: Thread block size exceeds maximum 1024")
        case ("LAUNCH", kernel, int(threads)) if threads > 1024:
            print("Error: Thread block size exceeds maximum 1024")
        case ("LAUNCH", kernel, (grid_x, grid_y), int(threads)):
            print(f"Launching {kernel} on ({grid_x}, {grid_y}) with {threads} threads")
        case ("LAUNCH", kernel, int(threads)):
            print(f"Launching {kernel} 1D grid with {threads} threads")
        case _:
            raise ValueError(f"Unknown instruction: {packet}")

Exercise 2: Nested Configuration Tree Parser
Write a function parse_pipeline(config) that processes tree structures:
configs = [
    {"stage": "LOAD", "source": "disk", "path": "/data/weights.bin"},
    {"stage": "TRANSFORM", "ops": ["NORMALIZE", "QUANTIZE"]},
    {"stage": "EXECUTE", "backend": {"device": "CUDA", "id": 0}},
    {"stage": "EXECUTE", "backend": {"device": "CPU"}},
]

In [ ]:
def parse_pipeline(config):
    match config:
        case {"stage":"LOAD","source":str(source),"path":str(path)}:
            print(f"stage: LOAD, source: {source}, path {path}")
        
        case {"stage":"TRANSFORM","ops":list(ops)}:
            print(f"stage: TRANSFORM, operations: {[op for op in ops]}")
        
        case {"stage":"EXECUTE", "backend": {"device":device,"id": id} }:
            print(f"stage: EXECUTE, device: {device}, id: {id}")
            
        case {"stage": "EXECUTE", "backend":{"device":device}}:
            print(f"STAGE:EXECUTE, device: {device}")